# 1) Import Configuration and Functions:

In [0]:
%run ../common/configuration


In [0]:
%run ../common/functions

## 2) Define Circuits Schema:

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

circuits_schema = StructType([
    StructField("circuit_id", StringType(), False),
    StructField("circuit_name", StringType(), True),
    StructField("url", StringType(), True),
    StructField("lat", DoubleType(), True),
    StructField("long", DoubleType(), True),
    StructField("locality", StringType(), True),
    StructField("country", StringType(), True),
])

circuits_input_path = f"{processed_folder_path}/circuits/csv/circuits.csv"

circuits_df = spark.read \
    .option("header", True) \
    .schema(circuits_schema) \
    .csv(circuits_input_path)



# 3) Transform Circuits Data:

The steps included:

- Drop column "url".
- Create Surrogate Key.
- Add Data Source and File Date.

In [0]:
from pyspark.sql.functions import lit

circuits_with_audit_df = circuits_df \
    .withColumn("data_source", lit(v_data_source)) \
    .withColumn("file_date", lit(v_file_date))

circuits_date_df = add_ingestion_date(circuits_with_audit_df)

circuits_dropped_df = circuits_date_df.drop("url")

circuits_final_df = add_surrogate_key(
    circuits_dropped_df,
    key_column_name="circuit_sk",
    hash_columns=["circuit_id", "circuit_name", "lat", "long", "locality", "country"],
)


print("Final columns going into the write:", circuits_final_df.columns)

# 4) Save the Processed Dataset to Delta Lake:

In [0]:
circuits_output_path = f"{processed_folder_path}/circuits/delta"

spark.sql("CREATE DATABASE IF NOT EXISTS f1_processed")

upsert_if_changed(
    input_df=circuits_final_df,
    db_name="f1_processed",
    table_name="circuits",
    output_path=circuits_output_path,
    merge_key_columns=["circuit_id"],
)

In [0]:
display(spark.read.format("delta").load(circuits_output_path))

In [0]:
build_presentation_dimension(
    processed_location=f"{processed_folder_path}/circuits/delta",
    natural_key_column="circuit_id",
    keep_columns=["circuit_id", "circuit_name", "lat", "long", "locality", "country"],
    presentation_directory=f"{presentation_folder_path}/dim_circuits/delta",
    db_name="f1_presentation",
    table_name="dim_circuits",
)

In [0]:
display(spark.read.format("delta").load(f"{presentation_folder_path}/dim_circuits/delta"))

# 5) Save backup Circuits in CSV format:

In [0]:
import io
import csv

circuits_backup_path = f"{presentation_folder_path}/dim_circuits/csv/dim_circuits.csv"

backup_rows = [row.asDict() for row in circuits_final_df.collect()]
backup_fieldnames = circuits_final_df.columns

backup_buffer = io.StringIO()
backup_writer = csv.DictWriter(backup_buffer, fieldnames=backup_fieldnames)
backup_writer.writeheader()
backup_writer.writerows(backup_rows)

dbutils.fs.put(circuits_backup_path, backup_buffer.getvalue(), overwrite=True)
print(f"backup saved: {circuits_backup_path}")